# SCD Type 2 with Change Data Feed pipeline test demonstration

In [0]:
dbutils.widgets.text("delta_catalog", "dbr_dev_ua5816bd")
dbutils.widgets.text("delta_schema", "oles0305_lab10")

In [0]:
delta_catalog = dbutils.widgets.get("delta_catalog")
delta_schema = dbutils.widgets.get("delta_schema")

base = f"{delta_catalog}.{delta_schema}"
src_table = f"{base}.customer_src"
tgt_table = f"{base}.customer_scd2"

cols = ["CustomerID", "FirstName", "LastName", "CompanyName", "EmailAddress"]

### Before changes

In [0]:
spark.sql(f"""
    SELECT {", ".join(cols)} 
    FROM {src_table} 
    WHERE CustomerID IN (1, 2, 3, 4, 99998, 99999)
""").display()

![](../screenshots/src_before.png)

In [0]:
spark.sql(f"""
    SELECT {", ".join(cols)}, valid_from, valid_to
    FROM {tgt_table} 
    WHERE CustomerID IN (1, 2, 3, 4, 99998, 99999)
""").display()

![](../screenshots/tgt_before.png)

### Simulate changes in `customer_src`

Test scenario

| CustomerID | Change | Expected result in `customer_scd2` |
|---|---|---|
| 1 | `EmailAddress` updated | old version closed, new current version |
| 2 | `CompanyName` set to `NULL` | old version closed, new current version (null-safe comparison) |
| 3 | update without real changes | no changes |
| 4 | deleted | current version closed, no new version |
| 99999 | inserted | new current version |
| 99998 | inserted and deleted before the load | not present |

In [0]:
spark.sql(f"UPDATE {src_table} SET EmailAddress = 'new@mail.com' WHERE CustomerID = 1")

spark.sql(f"UPDATE {src_table} SET CompanyName = NULL WHERE CustomerID = 2")

spark.sql(f"UPDATE {src_table} SET FirstName = FirstName WHERE CustomerID = 3")

spark.sql(f"DELETE FROM {src_table} WHERE CustomerID = 4")

spark.sql(f"""
    INSERT INTO {src_table} (CustomerID, FirstName, LastName, CompanyName, EmailAddress)
    VALUES (99999, 'Test', 'User', 'Co', 't@mail.com')
""")                                  

spark.sql(f"""
    INSERT INTO {src_table} (CustomerID, FirstName, LastName, CompanyName, EmailAddress)
    VALUES (99998, 'Tmp', 'User', 'Co', 'x@mail.com')
""") 

spark.sql(f"DELETE FROM {src_table} WHERE CustomerID = 99998") 

### After changes

In [0]:
spark.sql(f"""
    SELECT {", ".join(cols)} 
    FROM {src_table} 
    WHERE CustomerID IN (1, 2, 3, 4, 99998, 99999)
""").display()

![](../screenshots/src_after.png)

In [0]:
spark.sql(f"""
    SELECT {", ".join(cols)}, valid_from, valid_to
    FROM {tgt_table} 
    WHERE CustomerID IN (1, 2, 3, 4, 99998, 99999)
    ORDER BY CustomerID, valid_from
""").display()

![](../screenshots/tgt_after.png)